# Behavioral: 6k-Endpoint Telemetry Infrastructure

| Story | Core Signal |
|-------|-------------|
| Telemetry system design | Observability at scale, push vs pull |
| CloudWatch ETL pipeline | Data engineering + monitoring integration |
| Cost optimization | $X/month metrics cost → structured reduction |
| Data model for metrics | Time-series schema, aggregation strategy |
| Follow-up Defense | Metrics cardinality, sampling, SLO vs SLA |

**Purpose**: The telemetry story demonstrates your ability to design observability infrastructure for large-scale systems, make cost/coverage trade-offs, and own the full data lifecycle from collection to alerting.

```
Key concepts:
  Telemetry pillars: metrics, logs, traces
  Push vs pull: agents push metrics to collector vs collector scrapes
  Cardinality: high-cardinality labels → metric explosion → cost
  Aggregation: pre-aggregate at collection time to reduce storage
  SLO/SLI: service level objective (target) / indicator (measurement)
```

## Visual Model

```
TELEMETRY ARCHITECTURE (6k endpoints)
───────────────────────────────────────

  6,000 endpoints
      │
      │ push every 60s
      ▼
  CloudWatch Agent (per host) → CloudWatch Metrics
                                  │
                                  ├─ Dashboards (real-time)
                                  ├─ Alarms (threshold breach)
                                  └─ Metrics Streams → Kinesis → S3
                                                         │
                                                         └─ Athena (historical analysis)

BEFORE: naive design
  - Every raw metric per host per second stored in CloudWatch
  - 6,000 hosts × 50 metrics × 1/min = 300k metric data points/min
  - Cost: ~$12,000/month in CloudWatch

AFTER: structured collection
  - Pre-aggregate at agent: 1-min stats (avg, p95, p99, count)
  - CloudWatch for alerting only (< 20 metrics per host)
  - Full fidelity to S3 via Kinesis → Athena for analysis
  - Cost: ~$2,100/month (83% reduction)

CARDINALITY TRAP
  BAD:  metric{endpoint_id="uuid-1234", status_code="404", user_id="abc"}
        6k endpoints × 10 status codes × 1M users = 60B metric series
  GOOD: metric{endpoint_region="East", status_class="4xx"}
        6k × 5 regions × 5 status classes = 150k series
```

## Setup — Story Framework

In [ ]:
def print_story(title, situation, task, actions, result, timing_s=120):
    print(f"\n{'='*60}")
    print(f"STORY: {title}")
    print(f"Target: ~{timing_s}s verbal delivery")
    print(f"{'='*60}")
    print("\n[SITUATION] ~20s")
    print(situation)
    print("\n[TASK] ~10s")
    print(task)
    print("\n[ACTIONS] ~60s")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")
    print("\n[RESULT] ~30s")
    print(result)

def print_followup(q, a):
    print(f"\nQ: {q}")
    print(f"A: {a}")

print("Story framework loaded.")

## Decision Map — Telemetry Design

```
Telemetry design decisions
│
├─ What to collect?
│   Metrics: numeric, time-stamped, aggregatable (CPU, latency, error_rate)
│   Logs:    structured events, full context, high volume
│   Traces:  request flow across services (X-Ray, Jaeger)
│
├─ Push vs Pull?
│   Push: agents emit to central collector — good for ephemeral instances
│   Pull: collector scrapes endpoints (Prometheus) — good for stable services
│   Choice at 6k endpoints: push (instances may scale in/out; push guarantees delivery)
│
├─ Where to store?
│   Short-term (alerting):   CloudWatch, Datadog, Prometheus+Thanos
│   Long-term (analysis):    S3 (Parquet), BigQuery, TimescaleDB
│   Trade-off: retention cost vs query capability
│
├─ Cardinality budget?
│   Low-cardinality labels: region, tier, status_class (5xx/4xx)
│   High-cardinality labels: user_id, request_id → logs only, not metrics
│   Rule: labels that create >10k unique series → store in logs, not metrics
│
└─ Alerting strategy?
    SLO-based: alert when error budget burns faster than target rate
    Threshold: simple but brittle (alert on 5xx rate > 1%)
    Anomaly: ML-based baseline, alert on deviation > 2σ
    Choose: SLO for user-facing; threshold for infrastructure
```

## Pattern 1 — Telemetry Infrastructure: Core Story

In [ ]:
print_story(
    title="6k-Endpoint Telemetry Infrastructure — CloudWatch ETL",

    situation="""
At HorizonScale, our retail forecasting platform served ~6,000 endpoint agents
(in-store edge devices) that each emitted telemetry every 60 seconds.
When I joined, there was no structured observability — engineers SSH'd into
individual machines to debug issues, and the ops team had no visibility
into system health across the fleet. We had three P1 incidents in Q1 that
took 4+ hours to diagnose because we couldn't correlate signals.
""",

    task="""
I was tasked with designing and building the telemetry infrastructure
from scratch. This included collection, storage, cost optimization,
alerting, and a self-service dashboard for the operations team.
""",

    actions=[
        """First, I audited what data mattered. I interviewed the ops team and
   the SRE lead and identified 8 critical SLIs: CPU usage, memory, disk
   I/O, network latency to central API, sync success rate, forecast
   freshness (age of last received forecast), error rate, and heartbeat.
   I explicitly rejected 40+ other metrics as 'debug-only' — they'd be
   in logs, not metrics.""",

        """Deployed CloudWatch Agent to all 6,000 endpoints via SSM (Systems Manager
   Run Command). The agent was configured to pre-aggregate 1-minute stats
   (average, p95, p99, count) before sending — reducing cardinality from
   raw per-second samples to structured summaries. This was the key cost
   lever: we went from 300k data points/minute to 48k/minute.""",

        """Built a CloudWatch Metrics Streams → Kinesis Firehose → S3 pipeline
   for long-term retention. The S3 data was partitioned by date+region
   and stored as Parquet. This let the data science team run historical
   fleet health queries in Athena without paying CloudWatch query fees
   and enabled 13-month lookback for capacity planning.""",

        """Set up SLO-based alerting rather than raw thresholds. For the
   sync_success_rate SLI, the SLO was 99.5% over 30 days. I computed
   the error budget burn rate — if we were burning 2x faster than expected,
   an alert fired. This reduced alert fatigue by 70% vs the previous
   approach of alerting on any 5-minute dip below 99%.""",
    ],

    result="""
CloudWatch cost: $12,000/month → $2,100/month (83% reduction).
Time to diagnose P1 incidents: 4+ hours → 18 minutes average
  (ops team could now correlate fleet signals in one dashboard).
Alert fatigue: reduced by 70% with SLO-based burn rate alerting.
Fleet coverage: went from 0% to 100% in 3 weeks via SSM deployment.
Within 6 months, the data science team used the S3 metrics history
to identify that a specific firmware version caused a 12% increase
in sync failures — a correlation impossible to find without the data.
""",
    timing_s=150
)

## Pattern 2 — Metrics ETL Simulation

In [ ]:
# Simulate the metrics collection and ETL pipeline
# Demonstrates understanding of pre-aggregation, cardinality, partitioning

import random
import statistics
from collections import defaultdict
from datetime import datetime, timedelta

class MetricsAgent:
    """Simulates CloudWatch Agent: collects raw samples, emits 1-min aggregates."""

    def __init__(self, endpoint_id, region):
        self.endpoint_id = endpoint_id
        self.region = region
        self.buffer = defaultdict(list)  # metric_name → [sample, ...]

    def record(self, metric, value):
        self.buffer[metric].append(value)

    def flush(self, timestamp):
        """Aggregate 1 minute of samples into stats → emit to collector."""
        records = []
        for metric, values in self.buffer.items():
            if not values:
                continue
            sorted_v = sorted(values)
            n = len(sorted_v)
            records.append({
                "endpoint_id": self.endpoint_id,
                "region": self.region,
                "metric": metric,
                "timestamp": timestamp,
                "avg": round(statistics.mean(values), 3),
                "p95": round(sorted_v[int(0.95 * n)], 3),
                "p99": round(sorted_v[int(0.99 * n)], 3),
                "count": n,
                "min": round(sorted_v[0], 3),
                "max": round(sorted_v[-1], 3),
            })
        self.buffer.clear()
        return records


class MetricsETL:
    """Collects from all agents, partitions to S3-style storage."""

    def __init__(self):
        self.s3_partitions = defaultdict(list)  # (date, region, metric) → records
        self.cloudwatch_buffer = []  # last 2 weeks only (alerting)

    def ingest(self, records):
        for r in records:
            date_part = r["timestamp"][:10]
            partition = (date_part, r["region"], r["metric"])
            self.s3_partitions[partition].append(r)
            self.cloudwatch_buffer.append(r)

    def query_region(self, metric, region, date):
        """Fleet-wide P95 for a metric in a region on a given date."""
        rows = self.s3_partitions.get((date, region, metric), [])
        if not rows:
            return None
        return {
            "metric": metric, "region": region, "date": date,
            "fleet_avg_p95": round(statistics.mean(r["p95"] for r in rows), 3),
            "endpoints": len(rows)
        }


# Simulate 60 endpoints across 2 regions emitting 1 minute of data
random.seed(42)
etl = MetricsETL()
ts = "2024-03-15T10:00:00"

for i in range(60):
    region = "East" if i < 40 else "West"
    agent = MetricsAgent(f"EP{i:04d}", region)
    # Simulate 60 raw samples per minute (per-second)
    for _ in range(60):
        agent.record("cpu_pct", random.gauss(35, 12))
        agent.record("sync_latency_ms", random.expovariate(1/50))
    records = agent.flush(ts)
    etl.ingest(records)

print(f"S3 partitions created: {len(etl.s3_partitions)}")
print(f"Total aggregated records: {sum(len(v) for v in etl.s3_partitions.values())}")
print(f"(Without pre-aggregation: {60 * 60 * 2} raw samples would be stored)")

result = etl.query_region("cpu_pct", "East", "2024-03-15")
print(f"\nFleet P95 CPU (East, 2024-03-15): {result}")

print(f"\nCardinality comparison:")
print(f"  Raw (endpoint x metric x second): 60 × 2 × 60 = {60*2*60} data points")
print(f"  Pre-aggregated (endpoint x metric x minute): {len(etl.cloudwatch_buffer)} records")
print(f"  Reduction factor: {60*2*60 // len(etl.cloudwatch_buffer)}x")

## Pattern 3 — SLO-Based Alerting

In [ ]:
# SLO alerting: track error budget burn rate, not raw threshold
# Reduces alert fatigue: short blips don't page; sustained burn does

class SLOMonitor:
    """Monitors error budget consumption and burns."""

    def __init__(self, slo_target, window_days=30):
        self.slo_target = slo_target  # e.g., 0.995 = 99.5%
        self.window_days = window_days
        self.observations = []  # (timestamp, success_rate)

    @property
    def error_budget_pct(self):
        """Total error budget = 1 - SLO target."""
        return 1.0 - self.slo_target

    def record(self, timestamp, success_rate):
        self.observations.append((timestamp, success_rate))

    def burn_rate(self, window_minutes=60):
        """Burn rate = actual error rate / allowed error rate.
        Burn rate > 1 means consuming budget faster than the SLO allows.
        """
        recent = [r for _, r in self.observations[-window_minutes:]]
        if not recent:
            return 0.0
        avg_success = sum(recent) / len(recent)
        actual_error = 1.0 - avg_success
        allowed_error = self.error_budget_pct
        return actual_error / allowed_error if allowed_error > 0 else 0.0

    def budget_remaining(self):
        """Fraction of error budget remaining for this window."""
        if not self.observations:
            return 1.0
        total_allowed_errors = self.error_budget_pct * len(self.observations)
        actual_errors = sum(1.0 - r for _, r in self.observations)
        return max(0.0, 1.0 - actual_errors / total_allowed_errors)

    def alert(self, burn_threshold=2.0):
        br = self.burn_rate()
        budget_left = self.budget_remaining()
        status = "ALERT" if br > burn_threshold else "OK"
        return {
            "status": status,
            "burn_rate": round(br, 2),
            "budget_remaining_pct": round(budget_left * 100, 1),
        }


# Simulate: SLO = 99.5% sync success rate
monitor = SLOMonitor(slo_target=0.995)

print("=== Normal operation (99.7% success rate) ===")
for i in range(100):
    monitor.record(f"t{i}", 0.997)
print(monitor.alert())

print("\n=== Brief dip (99.0% for 5 min — threshold alert would fire) ===")
for i in range(5):
    monitor.record(f"t{100+i}", 0.990)
print(monitor.alert(burn_threshold=2.0))

print("\n=== Sustained degradation (95% for 60 min — burn rate alert fires) ===")
monitor2 = SLOMonitor(slo_target=0.995)
for i in range(60):
    monitor2.record(f"t{i}", 0.95)  # 5% error rate, 10x allowed
result = monitor2.alert(burn_threshold=2.0)
print(result)
print(f"\nAt 10x burn rate, 30-day budget consumed in: ~{30/10:.1f} days")

print("""
Why burn rate alerting:
  Threshold alert: fires on any 5-min dip → alert storm, pager fatigue
  Burn rate alert: only fires when budget consumption threatens the SLO
  A brief dip barely moves the 30-day needle — burn rate < 2x → no page
  Sustained outage burns budget fast — burn rate > 2x → page
""")

## Pattern 4 — Follow-up Defense: Telemetry

In [ ]:
followups = [
    (
        "Why CloudWatch instead of Prometheus + Grafana?",
        """At 6k endpoints on EC2, CloudWatch Agent was the natural fit:
it's IAM-native (no credentials to manage), integrates with SSM for
fleet-wide config deployment, and AWS Support covers it.
Prometheus would have required a scrape infrastructure (Prometheus servers,
service discovery, Thanos for long-term storage) — more operational overhead.
We evaluated both; for a team of 3 SREs, CloudWatch's managed approach
won on total operational cost. If we'd been running on Kubernetes,
Prometheus would have been the obvious choice."""
    ),
    (
        "What's metric cardinality and why does it matter?",
        """Cardinality is the number of unique label combinations for a metric.
High cardinality → exponential metric series → storage and query cost explosion.
For example: if you tag a metric with user_id and you have 1M users,
each metric creates 1M time series. At CloudWatch pricing of $0.30/metric/month,
that's $300k/month just for that one metric.
Our rule: any label with >10,000 unique values goes to logs (CloudWatch Logs
or S3), not metrics. High-cardinality analysis lives in Athena, not dashboards."""
    ),
    (
        "How did you handle endpoints that go offline?",
        """The heartbeat metric was our primary signal — each endpoint pushed a
heartbeat timestamp every 60 seconds. A CloudWatch alarm fired if heartbeat
age exceeded 5 minutes (missed 5 consecutive pushes).
We distinguished planned downtime (maintenance window tag in SSM) from
unplanned outages. Planned windows suppressed the alarm.
We also tracked fleet availability as an SLI: (endpoints heartbeating / total
registered) with a 99.0% SLO — less strict than the sync SLO since network
blips were expected at that scale."""
    ),
    (
        "How would you scale this to 600k endpoints?",
        """Three changes:
1. Hierarchical aggregation: instead of each endpoint pushing to CloudWatch
   directly, regional aggregators (one per AZ) pre-aggregate across
   endpoints before forwarding. This reduces CloudWatch PutMetricData calls
   by 100x per region.
2. Sampling: for non-critical metrics, sample 10% of endpoints per minute
   and extrapolate. For P1 signals (heartbeat, error rate), still collect all.
3. Move alerting off CloudWatch: at 600k scale, CloudWatch alarm costs
   become significant. Prometheus (self-hosted) or Grafana Cloud
   with remote write becomes cheaper for the alerting layer."""
    ),
    (
        "What's the difference between SLI, SLO, and SLA?",
        """SLI (indicator): the actual measurement. Example: 30-day sync success rate.
SLO (objective): the internal target we aim for. Example: SLI >= 99.5%.
SLA (agreement): the contractual commitment to customers. Example: 99.0% uptime.
The SLO should always be stricter than the SLA — you want to catch issues
before they breach the customer commitment. If SLO = 99.5% and SLA = 99.0%,
you have 0.5% as a buffer to catch and fix issues before they affect the SLA.
We never exposed SLI data directly in SLAs — that would remove the buffer."""
    ),
]

for q, a in followups:
    print_followup(q, a)

## Pattern 5 — Short-Form Variant Stories

In [ ]:
short_stories = {
    "Dashboard design / stakeholder story": """
After deploying telemetry, the ops team got a CloudWatch dashboard with
80 panels — everything I thought they might want to see.
Within a week, no one was using it. I sat with the lead ops engineer
and watched how they triaged incidents.
It turned out they needed 4 signals in one view: fleet heartbeat
health, sync failure rate, top-10 degraded endpoints, and active alarms.
I rebuilt it as a 4-panel dashboard. Adoption went from 0 to daily use.
Lesson: dashboard design starts with the operator's workflow, not the data.
""",

    "Metrics to S3 cost optimization": """
Our CloudWatch bill hit $12k/month in month 3 after we enabled raw
1-second metric collection for debugging a firmware issue.
I did a cost attribution query: 3 metrics from 6k endpoints at 1-second
resolution accounted for $9k of the bill.
I changed those 3 metrics to 1-minute pre-aggregated in the agent config,
and routed raw debug data to S3 (Kinesis Firehose, $0.029/GB) for offline analysis.
CloudWatch bill dropped to $2.1k the next month.
The debug data was available in S3 within 60 seconds via Athena — good enough.
""",

    "SSM fleet deployment story": """
Deploying CloudWatch Agent config to 6,000 endpoints could have been
a manual nightmare. I used AWS Systems Manager (SSM) Run Command
with a target filter: all instances tagged Environment=prod.
The deployment ran in parallel across 6k endpoints in under 8 minutes.
I also used SSM Parameter Store to centralize the agent config — a single
config update in Parameter Store triggered a re-read on all agents via
a cron + SSM document, no re-deployment needed.
This made config changes risk-free: rollback = revert the Parameter Store value.
""",

    "Incident: telemetry revealed firmware bug": """
Six months after deploying telemetry, the data science team ran a correlation
analysis on historical fleet metrics vs sync failure rate.
They found that endpoints on firmware v2.3.1 had a 12% higher
sync_latency_p99 and a 3x higher failure rate on rainy days (correlated
with humidity sensor data we'd also been collecting).
This was invisible without the longitudinal metrics history.
The vendor pushed a firmware fix in 6 weeks. Without the telemetry pipeline,
we'd never have found the pattern.
""",
}

for label, story in short_stories.items():
    print(f"\n{'─'*50}")
    print(f"SHORT STORY: {label}")
    print(f"{'─'*50}")
    print(story.strip())

## Full Decision Map

```
TELEMETRY SYSTEM DESIGN
─────────────────────────
What to instrument:
  SLIs first: what does the business care about?
  Infrastructure second: what do engineers need to debug?
  Debug data third: verbose, in logs/S3, not in metric system

Collection:
  Push (agent):   ephemeral fleet, dynamic scaling → CloudWatch Agent, StatsD
  Pull (scrape):  stable services, Kubernetes → Prometheus
  Pre-aggregate:  always — reduce data points before sending to metric store

Storage tiering:
  Hot (alerting):     CloudWatch / Datadog — last 2 weeks, expensive but fast
  Warm (trending):    TimescaleDB / InfluxDB — last 6 months
  Cold (analysis):    S3 + Athena — unlimited, cheap, queryable

Cardinality control:
  Labels with >10k unique values → logs, not metrics
  user_id, request_id, session_id → never in metric labels
  region, tier, status_class → OK metric labels

Alerting:
  SLO burn rate:  sustainable alerting, reduces fatigue
  Threshold:      simple, but noisy for variable traffic
  Anomaly:        ML baseline, good for unknown-unknown failures
  Page on:        user-facing SLI breach; team-level pager rotation
  Ticket on:      slow burn (budget < 50% remaining), non-urgent trends
```

## Cheat Sheet

```
TELEMETRY — KEY FACTS
───────────────────────
NUMBERS
  6,000 endpoints, 8 SLIs each
  Before: 300k raw data points/min → $12k/month
  After:  48k pre-aggregated/min → $2.1k/month (83% reduction)
  Incident MTTR: 4+ hours → 18 minutes
  Alert fatigue: -70% (SLO burn rate alerting)
  SSM fleet deploy: 6k endpoints in < 8 minutes

ARCHITECTURE
  CloudWatch Agent (pre-aggregate 1-min stats) → CloudWatch Metrics (alerting)
  CloudWatch Metrics Streams → Kinesis Firehose → S3 Parquet (analysis)
  Athena on S3 for historical fleet queries
  SSM Parameter Store for centralized agent config

SLO / SLI / SLA
  SLI: measurement (sync_success_rate = 99.7%)
  SLO: internal target (>= 99.5%)  ← stricter
  SLA: customer commitment (>= 99.0%)  ← looser
  Buffer: SLO - SLA = margin before customer breach

CARDINALITY RULE
  Low-cardinality labels → metrics: region, tier, status_class
  High-cardinality labels → logs: user_id, request_id, session_id
  >10k unique values → never in metric labels

BURN RATE ALERTING
  burn_rate = actual_error_rate / allowed_error_rate
  burn_rate > 2x → page  (consuming budget 2x faster than sustainable)
  burn_rate > 1x for 1h → ticket (trending toward breach)
```

## Summary Map

```
TELEMETRY INFRASTRUCTURE — ONE-PAGE SUMMARY
─────────────────────────────────────────────

CORE STORY
  S: 6k endpoints, no observability, P1 incidents take 4h+ to diagnose
  T: Design and build telemetry from scratch
  A: SLI audit (8 metrics, reject 40), CloudWatch Agent pre-aggregation,
     Kinesis→S3 long-term pipeline, SLO burn rate alerting, SSM deploy
  R: Cost -83%, MTTR 4h→18min, alert fatigue -70%, firmware bug found via history

DESIGN PRINCIPLES
  Instrument SLIs first, infrastructure second, debug-only → logs
  Pre-aggregate at the edge: reduce cardinality before sending
  Tier storage: hot (CloudWatch) / warm / cold (S3+Athena)
  Alert on burn rate, not thresholds

SHORT STORIES
  Dashboard: 80 panels → 4 panels; designed for operator workflow
  Cost spike: 1-sec metrics → pre-aggregated; $12k → $2.1k
  SSM deploy: 6k endpoints configured in 8 minutes
  Firmware bug: 6-month history revealed 12% latency increase on v2.3.1

INTERVIEW SIGNALS
  ✓ SLI/SLO/SLA definitions and the buffer concept
  ✓ Cardinality and why high-cardinality goes to logs
  ✓ Burn rate alerting vs threshold alerting
  ✓ Pre-aggregation at collection time for cost control
  ✓ Tiered storage for metrics (hot/cold)
  ✓ How telemetry enables debugging (historical correlation)
```